### <div style = 'background-color:#7cc458; color:#ffffff'> <center> Revisión Plan Mantenimiento Preventivo  </div>

In [1]:
import pandas as pd
import json
from datetime import timedelta, datetime
from dataclasses import dataclass, field
from typing import Optional

### <div style = 'color:#6ba84c'> <center> Definición de archivos y configuración </div>

In [2]:
mes = 6  
año = 2026 

#Nombres de los archivos  
archivoBlacklist     = "Site con Indisponibilidad casos complejos.xlsx"
archivoSWAP          = "REPORTE SWAPS EJECUTADOS NEW.xlsm"
archivoFrecuencias   = "Frecuencias.xlsx"
archivoEjecutados    =  "C:/Users/lcadillom/OneDrive - Integratel Perú/1_Compartido_KPI/BASE/V6.Autin -Segu mtto Preventivo Actualizado.xlsx"
archivoCM            = 'CM autin.xlsx'
archivoCMarchivados  = 'CM autin archivo.xlsx'
archivoPlanMPHuawei  = "C:/Users/lcadillom/OneDrive - Integratel Perú/SDC/Desempeño y Calidad/Plan 2026/Plan2026 - Huawei reduccion.xlsx"
archivoPlanMPComfica = "C:/Users/lcadillom/OneDrive - Integratel Perú/SDC/Desempeño y Calidad/Plan 2026/Plan2026 - Comfica reduccion.xlsx"
archivoOutput        = f"RevisionPlan - {mes}.xlsx"

#Nombres de las hojas 
hojaBlacklist     = "Data"
hojaSWAP          = "BASE"
hojaFrecuencias   = "Frecuencias"
hojaEjecutados    = "Data Preventivo"
hojaPlanMPHuawei  = "plan26 reducción modif"
hojaPlanMPComfica = "plan26 reduccion"
hojaCM            = 'Export All Custom'
hojaCMarchivados  = 'Export All Custom'

In [3]:
df_blacklist     = pd.read_excel(archivoBlacklist, sheet_name=hojaBlacklist)
df_SWAP          = pd.read_excel(archivoSWAP, sheet_name=hojaSWAP)
df_frecuencias   = pd.read_excel(archivoFrecuencias, sheet_name=hojaFrecuencias) #contiene la lista de todos los sitios 
df_ejecutados    = pd.read_excel(archivoEjecutados, sheet_name=hojaEjecutados)
df_planHuawei    = pd.read_excel(archivoPlanMPHuawei, sheet_name=hojaPlanMPHuawei)
df_planComfica   = pd.read_excel(archivoPlanMPComfica, sheet_name=hojaPlanMPComfica)
df_CMcompleto    = pd.read_excel(archivoCM, sheet_name=hojaCM)
df_CMarchivados  = pd.read_excel(archivoCMarchivados, sheet_name=hojaCMarchivados)

C:\Users\lcadillom\AppData\Roaming\Python\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\lcadillom\AppData\Roaming\Python\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [4]:
df_backlist     = df_blacklist[["C.U.", "FECHA REPORTE", "INGRESO A BLACKLIST", "ESTATUS"]]
df_blacklist    = df_blacklist.rename(columns={'C.U.': 'Site Id'})
df_SWAP         = df_SWAP[["CODIGO UNICO", "TIPO SITE", "ESTADO FINAL", "SWAP RAN REAL" ,"Despliegue"]]
df_SWAP         = df_SWAP.rename(columns={'CODIGO UNICO': 'Site Id'})
df_ejecutados   = df_ejecutados[["Site Id","Task Status", "Site Priority", "2_MES_PROGRA"]]
df_planHuawei["FLM"] = "Huawei"
df_planComfica["FLM"] = "Comfica"
df_plan         = pd.concat([df_planHuawei, df_planComfica])
df_plan         = pd.merge(df_plan, df_frecuencias, on='Site Id', how='left')
df_plan         = df_plan.rename(columns={"mes a ejecutar" : "MES_PROGRA"})
df_frecuencias  = df_plan.rename(columns={"tipo" : "tipo actual"})
estados_compl   = ['closed', 'completed']
df_CMarchivados = df_CMarchivados[df_CMarchivados['Task Status'].isin(estados_compl)]
df_CMcompleto   = df_CMcompleto[df_CMcompleto['Task Status'].isin(estados_compl)]
df_correctivos  = pd.concat([df_CMcompleto, df_CMarchivados])
df_correctivos  = df_correctivos[['Task Id', "Site Id", 'Complete Time']]

In [5]:
print(df_correctivos.columns)

Index(['Task Id', 'Site Id', 'Complete Time'], dtype='object')


### <div style = 'color:#6ba84c'> <center> Tratamiento de IDs Y Fechas   </div>

In [6]:
def normalizarIDs(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip().str.upper()

df_blacklist["Site Id"]     = normalizarIDs(df_blacklist["Site Id"])
df_ejecutados["Site Id"]    = normalizarIDs(df_ejecutados["Site Id"])
df_SWAP["Site Id"]          = normalizarIDs(df_SWAP["Site Id"])
df_plan["Site Id"]        = normalizarIDs(df_plan["Site Id"])

df_blacklist["INGRESO A BLACKLIST"] = df_blacklist["INGRESO A BLACKLIST"].astype(str).str.strip().str.upper()

df_plan["MES_PROGRA"] = pd.to_datetime({
    "year": 2026,
    "month": df_plan["MES_PROGRA"],
    "day": 1
})

meses = {
    "Ene": "Jan",
    "Feb": "Feb",
    "Mar": "Mar",
    "Abr": "Apr",
    "May": "May",
    "Jun": "Jun",
    "Jul": "Jul",
    "Ago": "Aug",
    "Sep": "Sep",
    "Oct": "Oct",
    "Nov": "Nov",
    "Dic": "Dec"
}

df_ejecutados["2_MES_PROGRA"] = (
    df_ejecutados["2_MES_PROGRA"]
    .astype(str)
    .replace(meses, regex=True)
)

df_ejecutados["2_MES_PROGRA"] = pd.to_datetime(
    df_ejecutados["2_MES_PROGRA"],
    format="%b-%y",
    errors="coerce"
)

### <div style = 'color:#6ba84c'> <center> Blacklist </div>

In [7]:
blacklist_ids = set(df_blacklist["Site Id"])
df_plan["blacklist"] = df_plan["Site Id"].apply(
    lambda s: "Si" if s in blacklist_ids else "No"
)

df_plan["revision"] = df_plan["blacklist"].apply(
    lambda x: "Excluir" if x == "Si" else ""
)

### <div style = 'color:#6ba84c'> <center> SWAP </div>

In [8]:
swap_map = df_SWAP.set_index("Site Id")[["SWAP RAN REAL", "Despliegue"]].to_dict(orient="index")

def get_swap_value(s):
    if s not in swap_map:
        return "No"
    
    swap_real = swap_map[s]["SWAP RAN REAL"]
    despliegue = swap_map[s]["Despliegue"]
    
    # Verificar si es fecha/hora con valor 00:00:00 o nulo
    swap_real_ts = pd.to_datetime(swap_real, errors="coerce")
    
    if pd.isnull(swap_real_ts) or swap_real_ts == pd.Timestamp("00:00:00"):
        fecha = despliegue
    else:
        fecha = swap_real_ts.strftime("%d/%m/%Y")
    
    return f"Sí ({fecha})"

df_plan["swap"] = df_plan["Site Id"].apply(get_swap_value)


### <div style = 'color:#6ba84c'> <center> Último mtto preventivo y correctivo </div>

In [9]:
df_ejecutados["2_MES_PROGRA"] = pd.to_datetime(df_ejecutados["2_MES_PROGRA"])
df_correctivos["Complete Time"] = pd.to_datetime(df_correctivos["Complete Time"])

df_ejecutados = df_ejecutados[df_ejecutados["Task Status"].str.lower().isin(["completed", "closed"])].copy()

ultimo_mp = (
    df_ejecutados.sort_values("2_MES_PROGRA")
    .groupby("Site Id")["2_MES_PROGRA"]
    .last()
    .dt.strftime("%d/%m/%Y")
    .to_dict()
)

ultimo_mc = (
    df_correctivos.sort_values("Complete Time")
    .groupby("Site Id")["Complete Time"]
    .last()
    .dt.strftime("%d/%m/%Y")
    .to_dict()
)

# Conteo de MCs por Site Id
conteo_mc = (
    df_correctivos
    .groupby("Site Id")["Complete Time"]
    .count()
    .to_dict()
)

df_plan["ultimo_mp"] = df_plan["Site Id"].map(ultimo_mp).fillna("Sin registro")
df_plan["ultimo_mc"] = df_plan["Site Id"].map(ultimo_mc).fillna("Sin registro")
df_plan["cantidad_mc"] = df_plan["Site Id"].map(conteo_mc).fillna(0).astype(int)

In [ ]:
def get_revision(row):
    #verificar si esta en blacklist 
    if row["blacklist"] == "Si":
        return "Excluir - Blacklist"
    
    # Verificar si tiene swap ejecutado en 2025
    swap_val = row["swap"]
    tiene_swap_2025 = "No" not in str(swap_val) and "2025" in str(swap_val)
    tiene_swap_2026 = "No" not in str(swap_val) and "2026" in str(swap_val)
    
    if tiene_swap_2025:
        if row["cantidad_mc"] < 2:
            return "Excluir - SWAP2025"
        else:
            return ""

    #v2> excluir siempre si se le realizo el swap 2026    
    if tiene_swap_2026:
        return "Excluir - SWAP2026"
        
    
    # Verificar frecuencia vs último MP
    ultimo_mp_str = row["ultimo_mp"]
    frecuencia = row["frecuencia "]
    mes_plan = row["MES_PROGRA"]
    
    if ultimo_mp_str != "Sin registro" and pd.notna(frecuencia) and pd.notna(mes_plan):
        try:
            ultimo_mp_dt = pd.to_datetime(ultimo_mp_str, format="%d/%m/%Y")
            mes_plan_dt = pd.to_datetime(mes_plan)
            
            meses_transcurridos = (
                (mes_plan_dt.year - ultimo_mp_dt.year) * 12
                + (mes_plan_dt.month - ultimo_mp_dt.month)
            )
            
            intervalo_meses = 12 / int(frecuencia)  
            
            if meses_transcurridos < intervalo_meses:
                return "Excluir - Frecuencia"
        except:
            pass
    
    return ""

df_plan["revision"] = df_plan.apply(get_revision, axis=1)

### <div style = 'color:#6ba84c'> <center> Exportar </div>

In [11]:
df_plan = df_plan[(df_plan['MES_PROGRA'].dt.month == mes)].copy()

df_plan['MES_PROGRA'] = mes
df_plan = df_plan.drop(["CLASIF HISPAM"	,"PDB",	"Supervición","Mes_Planif","Provincia",	"Tipo Zona_FLM","comentario","Unnamed: 10",	"Mes_Planif ini", "comentario"  ,"reprog mayo ","comentario "	], axis=1)

df_plan = df_plan[['Site Id', 'Site Name',"FLM", 'MES_PROGRA', 'tipo anterior', 'tipo',
       'frecuencia ', 'variacion', 'blacklist', 'swap',
       'ultimo_mp', 'ultimo_mc', 'cantidad_mc' , 'revision']]


#fechas = pd.to_datetime(df_plan["MES_PROGRA"], errors="coerce")

# df_plan["MES_PROGRA"] = (
#     fechas.dt.month
#     .eq(mes)
#     .where(lambda x: x, False)
#     .map({True: "Si", False: "No"})
# )
#df_plan = df_plan.rename(columns={"MES_PROGRA": "¿Programado este mes?"})

In [12]:
# Exportar a Exce
with pd.ExcelWriter(archivoOutput, engine='xlsxwriter') as writer:
    df_plan.to_excel(writer, sheet_name="revision", index=False)

In [14]:

# # ── 5. Exportar a JSON ────────────────────────────────────────────────────────
df_plan.to_json(
    "plan_mtto.json",
    orient="records",
    force_ascii=False,
    indent=2,
)

print(f"✓ {len(df_plan)} sitios exportados → plan_mtto.json")
print(df_plan.head())

✓ 302 sitios exportados → plan_mtto.json
     Site Id        Site Name     FLM  MES_PROGRA tipo anterior tipo  \
106  PI00131             JR_D  Huawei           6           B_2  B_2   
107  PI00139  AH.NVOHORIZONTE  Huawei           6           B_2  B_2   
108  PI00151         MORONA 2  Huawei           6           B_2  B_2   
109  PI00154           LA MAR  Huawei           6           B_2  B_2   
110  PI00156           TERELA  Huawei           6           B_2  B_2   

    frecuencia   variacion blacklist             swap     ultimo_mp  \
106           1  No Cambió        No  Sí (13/08/2025)  Sin registro   
107           1  No Cambió        No        Sí (2028)  Sin registro   
108           1  No Cambió        No  Sí (11/07/2025)  Sin registro   
109           1  No Cambió        No  Sí (17/07/2025)  Sin registro   
110           1  No Cambió        No  Sí (11/11/2025)  Sin registro   

      ultimo_mc  cantidad_mc revision  
106  13/02/2026            2           
107  12/03/2026    